# RepeatRadar Tutorial: Complete Guide to Cohort Analysis

**Welcome to RepeatRadar!** 🎯

This tutorial demonstrates the powerful cohort analysis capabilities of the `repeatradar` package. Whether you're analyzing user retention, revenue patterns, or any time-based metrics, this guide will show you everything you need to know.

## What You'll Learn:
- Basic user retention cohort analysis
- Revenue and value-based cohort analysis  
- Different time period configurations
- Retention rate calculations
- Output format options (pivot vs long format)
- Advanced aggregation functions
- Real-world examples and best practices

In [1]:
# Let's start by importing repeatradar and checking our setup
import repeatradar

print(f"🚀 RepeatRadar version: {repeatradar.__version__}")
print("📊 Ready for cohort analysis!")
print("\n" + "="*50)

🚀 RepeatRadar version: 0.4.1
📊 Ready for cohort analysis!



In [2]:
# Import the main function and required libraries
from repeatradar import generate_cohort_data
import pandas as pd
import numpy as np
from datetime import datetime

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', None)

This notebook demonstrates the `generate_cohort_data` function usage:

## 🎯 The Core Function: `generate_cohort_data()`

The `generate_cohort_data()` function is the heart of RepeatRadar. It transforms your transactional data into cohort analysis tables that reveal user behavior patterns over time.

### Key Features:
- **Flexible Time Periods**: Daily, weekly, monthly, quarterly, or yearly cohorts
- **Multiple Metrics**: User counts, revenue, averages, unique values, and more
- **Retention Rates**: Automatic percentage calculations
- **Complete Data**: No missing periods - gaps are filled with zeros
- **Dual Output Formats**: Pivot tables for visualization or long format for further analysis

Let's see it in action! 👇

## 📦 Sample Dataset: E-commerce Transactions

We'll use a real e-commerce dataset to demonstrate RepeatRadar's capabilities. This dataset contains customer transactions with purchase dates, customer IDs, and transaction values - perfect for cohort analysis!

In [3]:
# Load the sample e-commerce dataset
print("📥 Loading sample e-commerce dataset...")
ecommerce_data = pd.read_pickle("https://github.com/krinya/repeatradar/raw/refs/heads/main/examples/data/ecommerce_data_1.pkl")

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {ecommerce_data.shape[0]:,} transactions, {ecommerce_data.shape[1]} columns")
print(f"👥 Unique customers: {ecommerce_data['CustomerID'].nunique():,}")
print(f"📅 Date range: {ecommerce_data['InvoiceDateTime'].min().strftime('%Y-%m-%d')} to {ecommerce_data['InvoiceDateTime'].max().strftime('%Y-%m-%d')}")

📥 Loading sample e-commerce dataset...
✅ Dataset loaded successfully!
📊 Shape: 401,604 transactions, 10 columns
👥 Unique customers: 4,372
📅 Date range: 2010-12-01 to 2011-12-09


In [4]:
# Let's examine our dataset structure
print("📋 Dataset Overview:")
print(f"\nColumns: {list(ecommerce_data.columns)}")
print(f"\n📊 Sample Data:")
ecommerce_data.head()

📋 Dataset Overview:

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'InvoiceDateTime', 'TotalPrice']

📊 Sample Data:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateTime,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850,United Kingdom,2010-12-01 08:26:00,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01,2.75,17850,United Kingdom,2010-12-01 08:26:00,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34


## 🚀 Your First Cohort Analysis

### Basic User Retention Cohorts

Let's start with the most common type of cohort analysis: tracking how many unique users return in each period after their first purchase.

**Key Parameters:**
- `date_column`: Column containing transaction dates
- `user_column`: Column containing customer/user identifiers
- `cohort_period`: How to group acquisition periods ('M' = monthly)
- `period_duration`: How long each analysis period lasts (30 days)

In [5]:
# 📊 Basic Monthly User Cohort Analysis
# This shows how many unique users return in each 30-day period
print("🔍 Generating basic user retention cohorts...")

basic_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    cohort_period='M',       # Monthly cohorts (users grouped by acquisition month)
    period_duration=30       # Track in 30-day periods
)

print(f"✅ Generated cohort table: {basic_cohorts.shape[0]} cohorts × {basic_cohorts.shape[1]} periods")
print("\n📈 User Retention Cohort Table:")
basic_cohorts

🔍 Generating basic user retention cohorts...
✅ Generated cohort table: 13 cohorts × 13 periods

📈 User Retention Cohort Table:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,948,352,334,370,326,358,355,331,330,366,382,476,182
2011-01-01,421,107,118,127,126,124,102,113,139,144,131,13,0
2011-02-01,380,84,92,110,80,100,91,112,107,100,12,0,0
2011-03-01,440,81,106,100,89,89,110,117,97,20,0,0,0
2011-04-01,299,80,54,66,54,69,74,72,10,0,0,0,0
2011-05-01,279,56,42,56,69,61,78,6,0,0,0,0,0
2011-06-01,235,39,49,59,66,69,8,0,0,0,0,0,0
2011-07-01,191,34,40,46,46,2,0,0,0,0,0,0,0
2011-08-01,167,35,50,39,3,0,0,0,0,0,0,0,0


In [6]:
# 📊 Same Data in Long Format
# Long format is perfect for further analysis, plotting, or exporting
print("🔄 Converting to long format for analysis...")

long_format_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    cohort_period='M',
    period_duration=30,
    output_format='long'     # Key difference: long format instead of pivot
)

print(f"📊 Long format: {len(long_format_cohorts):,} rows")
print("\n📋 Sample of long format data:")
long_format_cohorts.head(10)

🔄 Converting to long format for analysis...
📊 Long format: 169 rows

📋 Sample of long format data:


,cohort_period,period_number,metric_value
0,2010-12-01,0,948
1,2010-12-01,1,352
2,2010-12-01,2,334
3,2010-12-01,3,370
4,2010-12-01,4,326
5,2010-12-01,5,358
6,2010-12-01,6,355
7,2010-12-01,7,331
8,2010-12-01,8,330
9,2010-12-01,9,366


In [7]:
# 📈 Automatic Retention Rate Calculation
# Convert raw user counts to retention percentages (% of users who return vs. Period 0)
print("🧮 Calculating retention rates as percentages...")

retention_rates = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    cohort_period='Q',                    # Quarterly cohorts for broader view
    period_duration=30,                   # Still track monthly periods
    calculate_retention_rate=True         # 🎯 This calculates percentages!
)

print(f"✅ Retention rates calculated!")
print("\n📊 Retention Rates (% of users returning each period):")
retention_rates

🧮 Calculating retention rates as percentages...
✅ Retention rates calculated!

📊 Retention Rates (% of users returning each period):


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-10-01,100.0,37.130001,35.230000,39.029999,34.389999,37.759998,37.450001,34.919998,34.810001,38.610001,40.299999,50.209999,19.200001
2011-01-01,100.0,21.920000,25.459999,27.160000,23.770000,25.219999,24.420000,27.559999,27.639999,21.270000,11.520000,1.050000,0.000000
2011-04-01,100.0,21.530001,17.840000,22.260000,23.250000,24.480000,19.680000,9.590000,1.230000,0.000000,0.000000,0.000000,0.000000
2011-07-01,100.0,23.930000,25.000000,13.870000,7.470000,0.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-10-01,100.0,12.460000,0.840000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### 🎨 Pro Tip: Formatting Your Results

For presentation purposes, you might want to round decimal values. Here's a handy utility function to do just that.

You can use this function to round any list of numerical results, making your data more readable and visually appealing. Just pass your results to the `round_results` function along with the desired number of decimal places, and it will return a new list with the rounded values.

In [8]:
from typing import Any, Optional

def format_cohort_values(x: Any, digits: Optional[int] = 1) -> Any:
    """
    Formats cohort values for better presentation.
    
    Args:
        x: The value to format
        digits: Number of decimal places for floats
    
    Returns:
        Formatted value (int for whole numbers, rounded float for decimals)
    """
    if isinstance(x, float):
        if x.is_integer():
            return int(x)  # Convert whole numbers to int
        return round(x, digits)
    if isinstance(x, str) and x.isdigit():
        return int(x) if int(x) == 0 else x
    return x

# Apply formatting to our retention rates
formatted_retention = retention_rates.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=1))
)

print("✨ Nicely formatted retention rates:")
formatted_retention

✨ Nicely formatted retention rates:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-10-01,100,37.1,35.2,39.0,34.4,37.8,37.5,34.9,34.8,38.6,40.3,50.2,19.2
2011-01-01,100,21.9,25.5,27.2,23.8,25.2,24.4,27.6,27.6,21.3,11.5,1.0,0.0
2011-04-01,100,21.5,17.8,22.3,23.2,24.5,19.7,9.6,1.2,0.0,0.0,0.0,0.0
2011-07-01,100,23.9,25.0,13.9,7.5,0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-10-01,100,12.5,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We can use same inputs for the period_duration as for the cohort period

## ⏰ Flexible Time Periods

### Using Period Shortcuts

Instead of specifying days as numbers, you can use convenient period shortcuts:
- `'D'` = Daily (1 day)
- `'W'` = Weekly (7 days)  
- `'M'` = Monthly (~30 days)
- `'Q'` = Quarterly (~90 days)
- `'Y'` = Yearly (~365 days)

This makes your code more readable and less error-prone!

In [9]:
# 📅 Weekly Analysis with Monthly Cohorts
# Track weekly user activity patterns for monthly acquisition cohorts
print("📅 Analyzing weekly patterns...")

weekly_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    period_duration='W',      # 🎯 Weekly periods (much cleaner than period_duration=7!)
    cohort_period='M'         # Monthly acquisition cohorts
)

print(f"📊 Weekly cohort analysis: {weekly_cohorts.shape[0]} cohorts × {weekly_cohorts.shape[1]} weeks")
weekly_cohorts

📅 Analyzing weekly patterns...
📊 Weekly cohort analysis: 13 cohorts × 54 weeks


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53
cohort_period,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-12-01,948,165,62,51,102,151,118,121,112,125,116,103,108,114,123,139,123,130,121,100,104,118,148,125,118,113,126,125,117,104,107,121,122,117,96,117,122,110,123,126,110,123,129,137,117,117,153,165,182,170,203,159,121,18
2011-01-01,421,32,32,28,27,36,29,40,29,33,40,29,31,31,33,35,52,31,51,39,34,32,34,40,36,29,31,34,39,27,33,37,33,29,42,40,41,37,50,35,43,47,43,59,50,42,34,15,3,0,0,0,0,0
2011-02-01,380,51,22,18,22,26,25,23,21,30,24,27,25,23,37,27,31,23,19,22,19,25,31,25,29,26,25,26,20,32,42,33,26,29,28,33,30,24,35,41,41,20,16,7,2,0,0,0,0,0,0,0,0,0
2011-03-01,440,22,28,25,24,20,30,22,37,28,32,29,30,32,31,27,27,28,32,24,24,22,24,32,25,28,29,32,33,24,27,41,40,37,34,39,39,26,14,17,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-04-01,299,26,14,15,23,24,26,21,19,17,14,11,12,20,18,23,13,14,19,18,13,13,25,21,20,15,18,25,18,25,21,27,25,14,10,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-05-01,279,25,21,16,22,16,11,10,13,12,14,14,14,17,16,14,16,19,13,13,25,24,17,13,20,15,29,26,23,17,6,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-06-01,235,15,18,13,14,13,7,11,13,15,12,13,17,23,15,13,17,17,18,24,16,29,26,30,12,10,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-07-01,191,13,13,17,8,10,13,10,8,14,8,15,9,12,16,11,16,16,15,15,10,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-08-01,167,10,17,8,14,13,7,13,16,10,19,14,12,15,15,10,5,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
# 🔍 Comparing Different Period Granularities
# See how different period durations affect your analysis scope

print("📊 Impact of Different Period Durations on Analysis Scope:")
print("=" * 60)

periods = [('D', 'Daily'), ('W', 'Weekly'), ('M', 'Monthly'), ('Q', 'Quarterly'), ('Y', 'Yearly')]

for period_code, period_name in periods:
    result = generate_cohort_data(
        data=ecommerce_data, 
        date_column='InvoiceDateTime', 
        user_column='CustomerID', 
        period_duration=period_code
    )
    print(f"{period_name:12} periods: {result.shape[0]:2} cohorts × {result.shape[1]:2} periods = {result.shape[0] * result.shape[1]:3} data points")

print("\n💡 Tip: More granular periods = more detailed insights but larger datasets!")

📊 Impact of Different Period Durations on Analysis Scope:
Daily        periods: 13 cohorts × 374 periods = 4862 data points
Weekly       periods: 13 cohorts × 54 periods = 702 data points
Monthly      periods: 13 cohorts × 13 periods = 169 data points
Quarterly    periods: 13 cohorts ×  5 periods =  65 data points
Yearly       periods: 13 cohorts ×  2 periods =  26 data points

💡 Tip: More granular periods = more detailed insights but larger datasets!


## 📈 Understanding Retention Rates

### What Are Retention Rates?

Retention rates show **what percentage** of users from each cohort return in subsequent periods, compared to the acquisition period (Period 0). 

- **Period 0** = 100% (all users by definition)
- **Period 1** = % of original users who return in period 1
- **Period 2** = % of original users who return in period 2
- And so on...

This makes it easy to compare cohort performance regardless of cohort size!

In [11]:
# 📈 Monthly Retention Rate Analysis
# Perfect for understanding long-term customer loyalty patterns
print("📈 Calculating monthly retention rates...")

monthly_retention = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    calculate_retention_rate=True,    # 🎯 Key parameter for percentage calculation
    period_duration='M'               # Monthly periods for business insights
)

print(f"✅ Monthly retention analysis complete!")
print(f"📊 Shape: {monthly_retention.shape[0]} cohorts × {monthly_retention.shape[1]} months")
print("\n📈 Monthly Retention Rates (% of original cohort):")
monthly_retention.round(1)  # Round to 1 decimal place for readability

📈 Calculating monthly retention rates...
✅ Monthly retention analysis complete!
📊 Shape: 13 cohorts × 13 months

📈 Monthly Retention Rates (% of original cohort):


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,100.0,37.099998,35.200001,39.000000,34.400002,37.799999,37.400002,34.900002,34.799999,38.599998,40.299999,50.200001,19.200001
2011-01-01,100.0,25.400000,28.000000,30.200001,29.900000,29.400000,24.200001,26.799999,33.000000,34.200001,31.100000,3.100000,0.000000
2011-02-01,100.0,22.100000,24.200001,29.000000,21.000000,26.299999,24.000000,29.500000,28.200001,26.299999,3.200000,0.000000,0.000000
2011-03-01,100.0,18.400000,24.100000,22.700001,20.200001,20.200001,25.000000,26.600000,22.000000,4.600000,0.000000,0.000000,0.000000
2011-04-01,100.0,26.799999,18.100000,22.100000,18.100000,23.100000,24.799999,24.100000,3.300000,0.000000,0.000000,0.000000,0.000000
2011-05-01,100.0,20.100000,15.000000,20.100000,24.700001,21.900000,28.000000,2.200000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-06-01,100.0,16.600000,20.799999,25.100000,28.100000,29.400000,3.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-07-01,100.0,17.799999,20.900000,24.100000,24.100000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-08-01,100.0,21.000000,29.900000,23.400000,1.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 🔍 Complete Data Coverage

### No More Missing Periods!

One of RepeatRadar's key features is **complete period coverage**. Traditional cohort analyses often have gaps where no activity occurred. RepeatRadar fills these gaps with zeros, giving you a complete picture.

**Why This Matters:**
- ✅ Perfect cohort triangles for visualization
- ✅ Consistent data structure for analysis
- ✅ No surprises when creating charts
- ✅ Easier to spot unusual patterns

In [12]:
# Long format showing complete periods (IMPROVED!)
long_format = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    output_format='long',
    period_duration='M'
)

print(f"Total rows in long format: {len(long_format)}")
print("\nSample of data including periods with 0 values:")
print(long_format.head(15))

print("\nPeriods with 0 values (previously these would be missing):")
zero_periods = long_format[long_format['metric_value'] == 0]
print(f"Found {len(zero_periods)} periods with 0 values")
print(zero_periods.head(10))

# 🔍 Demonstrating Complete Period Coverage
print("🔍 Examining data completeness...")

# Generate long format data to see the complete structure
complete_data = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    output_format='long',
    period_duration='M'
)

print(f"📊 Total data points: {len(complete_data):,}")
print(f"📅 Cohort periods: {complete_data['cohort_period'].nunique()}")
print(f"⏱️  Analysis periods: {complete_data['period_number'].nunique()}")

# Show periods with zero activity (these would be missing in other tools)
zero_periods = complete_data[complete_data['metric_value'] == 0]
print(f"\n🔍 Periods with zero activity: {len(zero_periods):,} ({len(zero_periods)/len(complete_data)*100:.1f}% of all periods)")
print("\n📋 Sample zero-activity periods:")
zero_periods.head(8)

Total rows in long format: 169

Sample of data including periods with 0 values:
   cohort_period  period_number  metric_value
0     2010-12-01              0           948
1     2010-12-01              1           352
2     2010-12-01              2           334
3     2010-12-01              3           370
4     2010-12-01              4           326
5     2010-12-01              5           358
6     2010-12-01              6           355
7     2010-12-01              7           331
8     2010-12-01              8           330
9     2010-12-01              9           366
10    2010-12-01             10           382
11    2010-12-01             11           476
12    2010-12-01             12           182
13    2011-01-01              0           421
14    2011-01-01              1           107

Periods with 0 values (previously these would be missing):
Found 78 periods with 0 values
   cohort_period  period_number  metric_value
25    2011-01-01             12             0
3

,cohort_period,period_number,metric_value
25,2011-01-01,12,0
37,2011-02-01,11,0
38,2011-02-01,12,0
49,2011-03-01,10,0
50,2011-03-01,11,0
51,2011-03-01,12,0
61,2011-04-01,9,0
62,2011-04-01,10,0


## 💰 Value-Based Cohort Analysis

### Beyond User Counts: Revenue Cohorts

While user retention is important, **revenue cohorts** show the financial value each cohort generates over time. This is crucial for understanding:

- 📈 Revenue per cohort over time
- 💎 Which cohorts are most valuable
- 📊 Average order values by cohort
- 🔄 Purchase frequency patterns

Let's explore different value-based analyses!

In [13]:
# 💰 Revenue Cohort Analysis
# Track total revenue generated by each cohort over time
print("💰 Analyzing revenue patterns by cohort...")

revenue_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',        # 🎯 The magic happens here!
    aggregation_function='sum',       # Sum all revenue per cohort/period
    period_duration='W',              # Weekly revenue tracking
    cohort_period='M'                 # Monthly acquisition cohorts
)

print(f"✅ Revenue analysis complete!")
print(f"📊 Shape: {revenue_cohorts.shape[0]} cohorts × {revenue_cohorts.shape[1]} weeks")
print(f"💵 Total revenue tracked: ${revenue_cohorts.sum().sum():,.2f}")
print("\n💰 Weekly Revenue by Cohort:")
revenue_cohorts.round(2)

💰 Analyzing revenue patterns by cohort...
✅ Revenue analysis complete!
📊 Shape: 13 cohorts × 54 weeks
💵 Total revenue tracked: $8,278,519.42

💰 Weekly Revenue by Cohort:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53
cohort_period,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-12-01,445345.83,72703.59,34858.45,26361.69,77739.28,60387.97,71351.48,47597.83,59587.90,70115.08,52538.46,41630.90,44089.20,64878.37,83836.39,78917.95,64705.32,50382.75,44345.36,46291.73,54938.25,54198.41,108821.85,59956.89,70739.76,56310.25,79767.38,72333.11,55081.88,77962.80,48669.04,70192.00,85201.39,115132.95,54465.69,67872.28,59621.51,72274.31,78366.06,97813.58,93383.96,149378.91,100115.20,133845.79,86074.81,108776.05,91404.91,117709.65,106180.25,128878.97,114107.58,88689.15,92181.51,13172.66
2011-01-01,197005.38,5099.75,5832.72,7990.71,11185.07,29096.90,7885.46,27086.82,9575.34,12604.86,13512.78,7878.15,12439.30,8614.68,15084.99,14029.83,18853.53,8533.50,35054.60,12333.81,14037.78,11142.30,35657.74,14435.50,11415.92,22854.88,13813.50,13968.49,17116.41,8638.91,10006.67,37533.51,13944.22,14798.19,18850.51,13619.23,19356.18,18431.25,43986.62,15054.19,25326.91,17786.08,14108.86,43165.90,30366.54,16026.83,12082.99,9488.78,1780.01,0.00,0.00,0.00,0.00,0.00
2011-02-01,137460.92,9355.27,4173.89,3277.71,7963.74,9616.74,6273.16,6059.57,9942.93,9793.60,11209.67,10188.91,7678.12,9099.40,12955.15,7637.68,12899.52,6205.83,8238.00,5098.73,5875.69,8106.15,8510.34,10147.97,8760.13,9644.96,11768.58,8888.21,6418.45,10598.29,21027.06,14910.75,11368.66,9110.03,11397.88,14912.97,10958.68,10461.79,15287.96,15963.80,13518.95,7433.87,4744.05,3000.79,871.22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-03-01,183265.85,3933.20,4812.37,5652.25,7709.02,5599.08,10938.48,5128.14,14585.75,12673.66,12360.11,9914.10,10476.06,9021.81,12496.72,9036.44,8303.04,10038.14,11395.48,9262.93,8604.49,8106.44,9929.96,12722.34,9944.76,12312.30,14633.08,13010.39,13428.25,12479.58,21799.72,14125.36,16947.74,10529.22,14657.04,15295.82,13075.96,7038.97,3504.55,3181.45,323.94,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-04-01,113312.46,4914.14,4866.90,3178.08,7331.49,7641.85,8383.54,5213.46,5220.90,7165.60,4679.34,4683.84,4257.81,5693.50,5173.51,8488.24,3418.26,7005.40,6031.91,5606.85,3293.94,8094.30,7921.23,7795.42,5050.02,3908.71,8777.95,9259.48,6027.48,8982.05,5096.83,9348.70,6598.75,4504.10,2750.62,1013.08,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-05-01,113041.70,1801.34,2640.15,2794.57,5615.62,3699.33,3214.34,3484.70,7007.71,3803.39,3806.59,6714.71,3065.85,3954.71,3741.88,3421.71,6847.42,4468.46,4670.25,5350.66,12424.74,10070.00,5344.05,3971.25,6028.28,4311.16,9321.73,7405.14,9061.58,8047.14,1445.01,424.04,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-06-01,90879.00,1562.79,2105.42,3918.19,1726.04,5039.26,1174.48,3994.55,2790.12,3211.09,2151.63,4126.21,3818.31,7692.60,10871.68,4732.85,7288.43,5327.43,8299.41,7590.10,3721.94,8849.73,12807.63,11321.03,2575.44,3379.66,1594.31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-07-01,62975.24,1697.07,1829.98,5570.84,1436.50,4074.46,4133.24,2154.76,1600.47,5272.69,3639.45,4087.42,2204.79,3732.95,5779.66,2638.35,4034.75,6691.86,3827.46,3228.00,2120.75,1553.24,-11.80,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-08-01,75678.06,235.34,2047.69,9780.81,5880.21,6032.38,4227.45,8639.85,9400.07,4016.62,16776.07,5472.73,17188.79,10287.11,4050.70,2202.55,597.51,1029.31,83.62,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## 🎯 Advanced Cohort Analyses

### Multiple Aggregation Functions

RepeatRadar supports various aggregation functions to analyze different aspects of your data:

- **`sum`**: Total values (revenue, quantities, etc.)
- **`mean`**: Average values (AOV, average session time, etc.)
- **`median`**: Median values (less affected by outliers)
- **`count`**: Count of records
- **`nunique`**: Count of unique values (products, categories, etc.)
- **`min`/`max`**: Minimum/maximum values

Let's explore some practical examples!

In [14]:
# 📊 Average Order Value (AOV) Analysis
# Understand spending patterns: are customers spending more or less over time?
print("📊 Calculating Order Value by cohort...")

sum_order_value = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='sum',      # 🎯 This will give revenue by period
    period_duration='M'
)

print(f"✅ Order value analysis complete!")
sum_order_value.round(0)

📊 Calculating Order Value by cohort...
✅ Order value analysis complete!


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,589533.0,269399.0,234009.0,311793.0,209192.0,298263.0,315586.0,332822.0,287861.0,459077.0,450069.0,491362.0,152319.0
2011-01-01,217959.0,76222.0,50863.0,60314.0,73463.0,83480.0,60478.0,86224.0,78521.0,94759.0,106629.0,9581.0,0.0
2011-02-01,158593.0,34748.0,39151.0,43758.0,28229.0,37402.0,41967.0,58181.0,58235.0,43298.0,5255.0,0.0,0.0
2011-03-01,202336.0,30529.0,51142.0,42600.0,43798.0,42943.0,58425.0,66458.0,49549.0,4473.0,0.0,0.0,0.0
2011-04-01,126914.0,31788.0,21765.0,24105.0,21973.0,30472.0,34361.0,26274.0,3038.0,0.0,0.0,0.0,0.0
2011-05-01,121959.0,20257.0,18214.0,18796.0,31262.0,23303.0,35338.0,1869.0,0.0,0.0,0.0,0.0,0.0
2011-06-01,99220.0,13094.0,13884.0,31829.0,28254.0,33658.0,2611.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-07-01,72826.0,11542.0,16064.0,16713.0,16932.0,194.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-08-01,88512.0,25411.0,44150.0,24440.0,1113.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# 📊 Average Order Value (AOV) Analysis
# Understand spending patterns: are customers spending more or less over time?
print("📊 Calculating Average Order Value by cohort...")

avg_order_value = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='mean',      # 🎯 Average instead of sum
    period_duration='M'
)

print(f"✅ AOV analysis complete!")
avg_order_value.round(2)

📊 Calculating Average Order Value by cohort...
✅ AOV analysis complete!


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,21.24,25.44,24.13,25.33,20.52,25.39,27.52,27.59,23.98,33.97,29.57,22.32,23.24
2011-01-01,18.23,27.60,17.53,18.23,23.67,26.66,23.11,23.66,18.53,19.65,18.81,35.10,0.00
2011-02-01,17.19,16.03,20.93,18.10,15.81,17.05,20.93,20.45,18.61,21.14,35.27,0.00,0.00
2011-03-01,17.15,18.74,21.33,16.56,19.43,20.80,19.36,17.72,14.85,9.22,0.00,0.00,0.00
2011-04-01,16.64,17.46,23.33,17.49,16.36,16.39,14.12,16.55,8.86,0.00,0.00,0.00,0.00
2011-05-01,18.99,19.37,21.18,16.56,15.32,14.86,15.79,20.77,0.00,0.00,0.00,0.00,0.00
2011-06-01,16.45,16.77,15.29,21.19,15.33,14.99,14.92,0.00,0.00,0.00,0.00,0.00,0.00
2011-07-01,13.71,14.09,12.44,11.96,10.22,14.93,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-08-01,15.13,11.01,14.73,14.28,21.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [16]:
# 📊 Median Order Value Analysis
# Median is less affected by outliers - great for understanding typical behavior
print("📊 Calculating Median Order Value (less affected by outliers)...")

median_order_value = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='median',    # 🎯 Median for robust analysis
    period_duration='M'
)

print(f"✅ Median analysis complete!")
print(f"💵 Overall median order value: ${median_order_value.median().median():.2f}")
print(f"💡 Compare: Average = ${avg_order_value.mean().mean():.2f}, Median = ${median_order_value.median().median():.2f}")
print("\n📈 Median Order Value by Cohort & Period:")
median_order_value.round(2)

📊 Calculating Median Order Value (less affected by outliers)...
✅ Median analysis complete!
💵 Overall median order value: $10.20
💡 Compare: Average = $10.27, Median = $10.20

📈 Median Order Value by Cohort & Period:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,11.10,12.60,11.80,11.25,11.70,13.00,10.50,10.50,11.70,13.20,12.60,10.14,9.9
2011-01-01,10.40,12.60,10.20,12.50,13.20,13.20,12.60,12.50,10.50,9.96,8.25,12.50,0.0
2011-02-01,13.20,10.30,14.85,15.00,12.48,13.20,15.00,15.00,14.85,15.00,16.50,0.00,0.0
2011-03-01,13.60,15.00,15.00,14.04,15.00,15.00,15.00,12.50,8.50,6.60,0.00,0.00,0.0
2011-04-01,14.85,13.50,15.30,14.85,11.34,13.50,10.50,12.75,5.90,0.00,0.00,0.00,0.0
2011-05-01,15.00,10.88,16.50,15.00,12.75,10.20,13.16,16.97,0.00,0.00,0.00,0.00,0.0
2011-06-01,10.50,8.40,12.55,15.00,10.08,7.82,10.20,0.00,0.00,0.00,0.00,0.00,0.0
2011-07-01,10.50,10.20,9.95,7.50,4.95,15.60,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2011-08-01,10.50,6.60,9.87,9.81,17.55,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


In [17]:
# 🧾 Transaction Frequency Analysis
# How many unique transactions does each cohort generate?
print("🧾 Analyzing transaction frequency by cohort...")

transaction_frequency = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='InvoiceNo',
    aggregation_function='nunique',   # 🎯 Count unique transactions
    period_duration='M'
)

print(f"✅ Transaction frequency analysis complete!")
print(f"🧾 Total unique transactions: {transaction_frequency.sum().sum():,}")
print("\n📈 Unique Transactions per Cohort & Period:")
transaction_frequency

🧾 Analyzing transaction frequency by cohort...
✅ Transaction frequency analysis complete!
🧾 Total unique transactions: 22,190

📈 Unique Transactions per Cohort & Period:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,1805,672,633,737,624,780,714,676,650,770,832,1103,323
2011-01-01,616,171,178,202,195,176,182,175,210,232,250,16,0
2011-02-01,545,130,132,156,109,141,122,163,160,135,18,0,0
2011-03-01,616,123,165,154,134,126,146,193,168,29,0,0,0
2011-04-01,421,124,65,83,74,92,110,99,16,0,0,0,0
2011-05-01,413,72,59,81,90,80,119,9,0,0,0,0,0
2011-06-01,339,51,72,79,97,109,9,0,0,0,0,0,0
2011-07-01,268,49,53,66,72,2,0,0,0,0,0,0,0
2011-08-01,232,58,80,52,4,0,0,0,0,0,0,0,0


In [18]:
# unique invoice_count
invoice_count= generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='InvoiceNo',
    aggregation_function='nunique',
    period_duration='M'
)

invoice_count

period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,1805,672,633,737,624,780,714,676,650,770,832,1103,323
2011-01-01,616,171,178,202,195,176,182,175,210,232,250,16,0
2011-02-01,545,130,132,156,109,141,122,163,160,135,18,0,0
2011-03-01,616,123,165,154,134,126,146,193,168,29,0,0,0
2011-04-01,421,124,65,83,74,92,110,99,16,0,0,0,0
2011-05-01,413,72,59,81,90,80,119,9,0,0,0,0,0
2011-06-01,339,51,72,79,97,109,9,0,0,0,0,0,0
2011-07-01,268,49,53,66,72,2,0,0,0,0,0,0,0
2011-08-01,232,58,80,52,4,0,0,0,0,0,0,0,0


## 🎉 Congratulations!

### You've Mastered RepeatRadar's Core Features!

You now know how to:

✅ **Create basic user retention cohorts**  
✅ **Calculate retention rates as percentages**  
✅ **Analyze revenue and value patterns**  
✅ **Use different time periods and granularities**  
✅ **Apply various aggregation functions**  
✅ **Work with both pivot and long data formats**  
✅ **Handle missing periods automatically**  

### 🚀 Next Steps

1. **Try with your own data**: Replace our sample dataset with your transaction data
2. **Experiment with parameters**: Different cohort periods, aggregation functions, etc.
3. **Create visualizations**: Use the pivot format data with plotting libraries
4. **Combine analyses**: Compare user retention vs. revenue cohorts

### 📚 Key Parameters Reference

| Parameter | Description | Examples |
|-----------|-------------|----------|
| `date_column` | Column with transaction dates | `'purchase_date'`, `'created_at'` |
| `user_column` | Column with user/customer IDs | `'customer_id'`, `'user_id'` |
| `value_column` | Column with values to analyze | `'revenue'`, `'quantity'`, `'product_id'` |
| `aggregation_function` | How to aggregate values | `'sum'`, `'mean'`, `'count'`, `'nunique'` |
| `cohort_period` | Cohort grouping | `'D'`, `'W'`, `'M'`, `'Q'`, `'Y'` |
| `period_duration` | Analysis period length | `'D'`, `'W'`, `'M'`, `30`, `7` |
| `output_format` | Result format | `'pivot'`, `'long'` |
| `calculate_retention_rate` | Calculate percentages | `True`, `False` |

---

**Happy analyzing!** 📊✨